# Model training

Compare repayment-status encodings and tune Logistic Regression, Random Forest,
and XGBoost using the same five development folds. Evaluate the previously
selected XGBoost model on the held-out test set at threshold 0.5.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))


from credit_default.paths import RAW_DATA_FILE
from credit_default.model import (
    CATEGORICAL_COLUMNS, DECISION_THRESHOLD, NUMERICAL_COLUMNS,
    RANDOM_STATE, REPAYMENT_COLUMNS, TARGET, TEST_SIZE, group_education_codes,
)

N_FOLDS = 5

## Load modeling data

Read the `Data` sheet using its second row as the header, normalize column names,
and exclude the identifier and target from the predictors. Dataset exploration is
documented in the EDA notebook.

In [2]:
df = pd.read_excel(RAW_DATA_FILE, sheet_name="Data", header=1, engine="calamine")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
X = df.drop(columns=["id", TARGET])
y = df[TARGET]

assert X.shape[1] == 23
assert X.index.equals(y.index)

## Reserve the test set and define shared folds

Keep the original 80/20 split and seed used by `train.py`. Use all development
records for five-fold stratified cross-validation; no additional holdout is needed
for the baseline. Generate the fold indices once for every comparison and search.

In [3]:
X_full_train, X_test, y_full_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
cv_splits = list(StratifiedKFold(
    n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE,
).split(X_full_train, y_full_train))

assert X_full_train.index.intersection(X_test.index).empty
assert X_full_train.index.equals(y_full_train.index)
assert X_test.index.equals(y_test.index)

display(pd.DataFrame({
    "Records": [len(y_full_train), len(y_test)],
    "Default rate": [y_full_train.mean(), y_test.mean()],
}, index=["Development", "Test"]))

,Records,Default rate
Development,24000,0.221792
Test,6000,0.218833


## Define preprocessing and model pipelines

Each pipeline groups education codes 5 and 6 into 0, One-Hot Encodes categorical
features, and scales numerical features only for Logistic Regression. Encoding
and scaling are fitted inside each cross-validation fold.

Compare numerical and categorical representations of the six repayment-status
(`pay_*`) features for Logistic Regression. Trees retain numerical repayment
codes and use no scaling. These are modeling choices; the meanings of codes -2
and 0 are not established by the dataset documentation.

In [4]:
base_categorical_cols = CATEGORICAL_COLUMNS
repayment_cols = REPAYMENT_COLUMNS
base_numerical_cols = [col for col in NUMERICAL_COLUMNS if col not in repayment_cols]


def build_pipeline(model, categorical_cols, numerical_cols, scale_numerical=False):
    configured = categorical_cols + numerical_cols
    assert len(configured) == len(set(configured))
    assert set(configured) == set(X.columns)

    preprocessing = ColumnTransformer([
        ("categorical", OneHotEncoder(
            handle_unknown="ignore", sparse_output=False,
        ), categorical_cols),
        ("numerical", StandardScaler() if scale_numerical else "passthrough",
         numerical_cols),
    ], remainder="drop")

    return Pipeline([
        ("education_recoding", FunctionTransformer(
            group_education_codes, feature_names_out="one-to-one",
        )),
        ("preprocessing", preprocessing),
        ("model", model),
    ])

In [5]:
numerical_pipeline = build_pipeline(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    base_categorical_cols, base_numerical_cols + repayment_cols,
    scale_numerical=True,
)
categorical_pipeline = build_pipeline(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    base_categorical_cols + repayment_cols, base_numerical_cols,
    scale_numerical=True,
)
rf_pipeline = build_pipeline(
    RandomForestClassifier(
        n_estimators=200, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1,
    ),
    base_categorical_cols, base_numerical_cols + repayment_cols,
)
xgb_pipeline = build_pipeline(
    XGBClassifier(
        objective="binary:logistic", eval_metric="auc", tree_method="hist",
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=2,
    ),
    base_categorical_cols, base_numerical_cols + repayment_cols,
)

## Shared tuning procedure

Rank candidates by validation ROC-AUC on the shared folds and refit the best
pipeline on all development records. Tables include training ROC-AUC and CV mean
and standard deviation. All standard deviations use `ddof=0`, as in `GridSearchCV`.

In [6]:
def tune(pipeline, param_grid):
    search = GridSearchCV(
        pipeline, param_grid, scoring="roc_auc", cv=cv_splits,
        return_train_score=True, refit=True, error_score="raise", n_jobs=1,
    )
    search.fit(X_full_train, y_full_train)

    results = pd.DataFrame(search.cv_results_)
    columns = [f"param_{parameter}" for parameter in param_grid] + [
        "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score",
    ]
    labels = {f"param_{parameter}": parameter.removeprefix("model__")
              for parameter in param_grid}
    labels.update({
        "mean_train_score": "Train ROC-AUC", "mean_test_score": "CV ROC-AUC",
        "std_test_score": "CV SD", "rank_test_score": "Rank",
    })
    display(results[columns].sort_values("rank_test_score").rename(columns=labels))
    return search

## Logistic Regression: encoding and regularization

Evaluate numerical repayment codes with `C=1` and tune `C` for categorical
repayment codes. Reuse the categorical search's `C=1` fold scores for the encoding
comparison, keeping regularization identical without fitting that baseline again.

In [7]:
numerical_cv = cross_validate(
    numerical_pipeline, X_full_train, y_full_train,
    cv=cv_splits, scoring="roc_auc", error_score="raise",
)
logreg_search = tune(categorical_pipeline, {
    "model__C": [0.01, 0.1, 1, 10, 100],
})

,C,Train ROC-AUC,CV ROC-AUC,CV SD,Rank
1,0.10,0.772891,0.768591,0.006839,1
2,1.00,0.772992,0.768255,0.006955,2
0,0.01,0.771557,0.768230,0.006311,3
4,100.00,0.773189,0.767939,0.007155,4
3,10.00,0.773149,0.767902,0.006624,5


In [8]:
categorical_baseline_index = logreg_search.cv_results_["params"].index({"model__C": 1})
encoding_fold_scores = pd.DataFrame({
    "Numerical": numerical_cv["test_score"],
    "Categorical": [
        logreg_search.cv_results_[f"split{fold}_test_score"][categorical_baseline_index]
        for fold in range(N_FOLDS)
    ],
}, index=pd.Index(range(1, N_FOLDS + 1), name="Fold"))
encoding_summary = pd.DataFrame({
    "CV ROC-AUC": encoding_fold_scores.mean(),
    "CV SD": encoding_fold_scores.std(ddof=0),
})
display(encoding_summary)

,CV ROC-AUC,CV SD
Numerical,0.722513,0.005455
Categorical,0.768255,0.006955


## Random Forest

Keep 200 trees and square-root feature sampling. Tune tree depth and minimum leaf
size using the same development folds.

In [9]:
rf_search = tune(rf_pipeline, {
    "model__max_depth": [10, None],
    "model__min_samples_leaf": [5, 20],
})

,max_depth,min_samples_leaf,Train ROC-AUC,CV ROC-AUC,CV SD,Rank
3,None,20,0.887621,0.783634,0.005005,1
1,10,20,0.832223,0.782130,0.006039,2
0,10,5,0.854582,0.781371,0.006392,3
2,None,5,0.968432,0.778205,0.005263,4


## XGBoost

Tune boosting rounds, tree depth, and learning rate. Keep row and column sampling
at 0.8 and minimum child weight at 5. Each candidate uses fixed boosting rounds;
validation remains inside cross-validation, with no external evaluation set.

In [10]:
xgb_search = tune(xgb_pipeline, {
    "model__n_estimators": [100, 300],
    "model__max_depth": [3, 6],
    "model__learning_rate": [0.05, 0.1],
})

,n_estimators,max_depth,learning_rate,Train ROC-AUC,CV ROC-AUC,CV SD,Rank
2,100,6,0.05,0.838083,0.783090,0.004536,1
1,300,3,0.05,0.815587,0.782412,0.004157,2
4,100,3,0.10,0.805189,0.781569,0.003914,3
6,100,6,0.10,0.869329,0.780159,0.004425,4
5,300,3,0.10,0.835604,0.779460,0.004139,5
0,100,3,0.05,0.793693,0.779448,0.005057,6
3,300,6,0.05,0.897358,0.778039,0.004553,7
7,300,6,0.10,0.942597,0.767565,0.004502,8


## Compare tuned models

Summarize each search's selected candidate. These development scores were also
used for tuning; the held-out test is evaluated separately below.

In [11]:
searches = {
    "Logistic Regression": logreg_search,
    "Random Forest": rf_search,
    "XGBoost": xgb_search,
}
model_comparison = pd.DataFrame([
    {
        "Model": name,
        "Selected parameters": {
            key.removeprefix("model__"): value
            for key, value in search.best_params_.items()
        },
        "Train ROC-AUC": search.cv_results_["mean_train_score"][search.best_index_],
        "CV ROC-AUC": search.best_score_,
        "CV SD": search.cv_results_["std_test_score"][search.best_index_],
    }
    for name, search in searches.items()
]).set_index("Model")
display(model_comparison)

,Selected parameters,Train ROC-AUC,CV ROC-AUC,CV SD
Model,,,,
Logistic Regression,{'C': 0.1},0.772891,0.768591,0.006839
Random Forest,"{'max_depth': None, 'min_samples_leaf': 20}",0.887621,0.783634,0.005005
XGBoost,"{'learning_rate': 0.05, 'max_depth': 6, 'n_est...",0.838083,0.783090,0.004536


## Evaluate the selected model on the held-out test

Retain the previously established XGBoost choice and threshold 0.5. The search has
already refitted the selected pipeline on all development records. Evaluate it
once without retuning the model or threshold from test outcomes. Report ROC-AUC
from probabilities and precision, recall, F1, and the confusion matrix from the
fixed decision threshold.

In [12]:
final_pipeline = xgb_search.best_estimator_
positive_class_index = list(final_pipeline.classes_).index(1)
test_probabilities = final_pipeline.predict_proba(X_test)[:, positive_class_index]
test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype(int)

final_test_metrics = pd.DataFrame([{
    "Model": "XGBoost",
    "Threshold": DECISION_THRESHOLD,
    "Test records": len(y_test),
    "ROC-AUC": roc_auc_score(y_test, test_probabilities),
    "Precision": precision_score(y_test, test_predictions, zero_division=0),
    "Recall": recall_score(y_test, test_predictions, zero_division=0),
    "F1": f1_score(y_test, test_predictions, zero_division=0),
}])
test_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=[0, 1]),
    index=["Actual no default (0)", "Actual default (1)"],
    columns=["Predicted no default (0)", "Predicted default (1)"],
)
display(final_test_metrics)
display(test_confusion_matrix)

,Model,Threshold,Test records,ROC-AUC,Precision,Recall,F1
0,XGBoost,0.5,6000,0.7839,0.67234,0.361005,0.469772


,Predicted no default (0),Predicted default (1)
Actual no default (0),4456,231
Actual default (1),839,474
